# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My data contract

### Unit of analysis

One row represents the daily performance of one content item for one client on one report date.

The expected grain is:

`report_date × client_hash_id × content_hash_id`

### Table

I use the `fact_content_daily_performance` table, specifically its March 2026 partition.

### Time window

The development window is March 2026. I use this mid-panel month rather than June 2026 because June is the final month and should be treated as a sealed outcome/test period.

### What I would predict/rank

I want to rank content items by their likelihood of needing attention based on their future performance trend. The ranking would help a content team decide which content items should be reviewed first.

### Deliberately excluded

I deliberately exclude future outcome information and the identifier fields `client_hash_id` and `content_hash_id` from model features. The identifiers are used only for grouping, joining, and splitting.

In [34]:
from huggingface_hub import HfFileSystem
import duckdb
import pandas as pd

fs = HfFileSystem()

print("Hugging Face connection successful!")



Hugging Face connection successful!


In [35]:
remote_path = (
    "datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

local_path = "march_2026.parquet"

fs.get(remote_path, local_path)

print("Downloaded successfully!")
print(local_path)

Downloaded successfully!
march_2026.parquet


In [36]:
con = duckdb.connect()

march = con.sql("""
    SELECT *
    FROM read_parquet('march_2026.parquet')
    LIMIT 5
""").df()

march

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## 2) Fields: feature / label / context / excluded

- **Feature:** knowable before the decision and safe to use.
- **Label / proxy:** the outcome being predicted or ranked; never a feature.
- **Context:** identifiers and fields used for grouping, joining, or splitting.
- **Excluded:** future information or fields whose timing/definition makes them unsafe.

The code below inspects the real March 2026 schema before selecting fields.


In [37]:
schema = con.sql("""
    DESCRIBE SELECT *
    FROM read_parquet('march_2026.parquet')
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3) Three required verification queries

The three required queries below verify:

1. the grain,
2. row count and date span,
3. availability using `IS TRUE`.

The schema/candidate cells above are setup cells and are not counted as the three verification queries.


### Verification 1 — Grain

The expected grain is `report_date × client_hash_id × content_hash_id`.

I will check for duplicate combinations of these three fields. If the query returns zero rows, there are no duplicate combinations in the March partition and the expected grain holds.

In [ ]:
grain_check = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_counta
    FROM read_parquet('march_2026.parquet')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

,report_date,client_hash_id,content_hash_id,row_count


### Verification 2 — Row count and date span

This query checks how many rows are in the March 2026 slice and the earliest and latest report dates represented in it.

In [39]:
count_dates = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('march_2026.parquet')
""").df()

count_dates

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### Verification 3 — Data availability

The warehouse contains separate availability flags for Search Console and Google Analytics.

I use `IS TRUE` rather than assuming that a zero-valued metric means data is available.

In [40]:
availability_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet('march_2026.parquet')
""").df()

availability_check

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## Five features

For the Ranking / Recommendation lane, I will start with five features that describe previously observed search and engagement performance.

The five features are:

1. `gsc_impressions`
2. `gsc_clicks`
3. `gsc_avg_position`
4. `ga4_sessions`
5. `ga4_engaged_sessions`

The identifiers and report date are context fields, not model features.

In [41]:
features = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions
    FROM read_parquet('march_2026.parquet')
    LIMIT 100
""").df()

features

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
...,...,...,...,...,...,...,...,...
95,2026-03-01,client_73cda7b4e4f265ea,content_70cb674f0bb80edc,4,0,43.500000,<NA>,<NA>
96,2026-03-01,client_73cda7b4e4f265ea,content_be74499afeb56f29,0,0,NaN,<NA>,<NA>
97,2026-03-01,client_73cda7b4e4f265ea,content_5f5b70c39b2bf746,0,0,NaN,<NA>,<NA>
98,2026-03-01,client_73cda7b4e4f265ea,content_a2bc04ce72550638,3,0,37.666667,<NA>,<NA>


### When is each feature available?

**`gsc_impressions`**  
Knowable at the decision moment because it comes from previously collected Search Console performance data.

**`gsc_clicks`**  
Knowable at the decision moment because it comes from previously collected Search Console performance data.

**`gsc_avg_position`**  
Knowable at the decision moment because it summarizes previously observed search ranking performance.

**`ga4_sessions`**  
Knowable at the decision moment because it comes from previously collected Analytics performance data when `ga4_data_available IS TRUE`.

**`ga4_engaged_sessions`**  
Knowable at the decision moment because it comes from previously collected Analytics engagement data when `ga4_data_available IS TRUE`.

## 4) Data limits

One limitation is that history depth differs substantially between clients, so a single calendar window may not represent the same amount of historical information for every client.

Another limitation is availability: rows before a client's GA4 start can contain zero-filled GA4 values with `ga4_data_available = FALSE`, so zero must not automatically be interpreted as zero engagement.

The March 2026 slice is historical, so patterns found in it may not generalize perfectly to future content or changing user behavior.


In [42]:
[c for c in schema["column_name"].tolist()
 if "trend" in c.lower() or "label" in c.lower()]

[]

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.